# Colab: teacher training → student KD + rotation self-distillation

1. Upload this notebook to Colab. Select Runtime → Change runtime type → GPU.
2. Run the single code cell. Authorize your Google Drive when prompted.
3. Select data-DL.zip in the upload prompt. Alternatively put data-DL.zip at MyDrive/data-DL.zip before starting; it will be copied to local disk automatically.
4. Teacher trains first (300 epochs), then student (330 epochs). Smoke test is disabled. No existing weights required. Default 45k train/5k validation split shared by both stages. Student including auxiliary head: 4,896,418 parameters.
5. Checkpoints and CSV outputs persist under MyDrive/Operation90_TwoStage/. Each epoch overwrites last.pth atomically. Keep this folder for resuming. Local extracted data/cache disappear when a runtime is reset and will be rebuilt.
6. After disconnection, reconnect GPU, rerun the same cell, remount the SAME Drive, and provide the ZIP again if it is not in MyDrive. Existing matching checkpoints resume automatically. Keep configuration unchanged. A completed teacher is restored and exported before the student resumes. Changing STAGE to student can skip reloading the teacher training stage once its export exists.

No automatic submission or package installation. Uses one GPU. Runtime availability/session duration are not guaranteed; preserve checkpoints. Do not change the epoch count when resuming an existing schedule. No accuracy improvement over 89.6% has been measured. Syntax checked; execution on Colab GPU not verified here.


In [ ]:
from __future__ import annotations
"""CIFAR-10 A100 single-student upgrade: optional warm-start + independent WRN challenger.

Inputs: provided competition images (not external CIFAR data) and a permitted
training-only WRN-28-10 checkpoint. The submitted student is always <5M params.

The default is a full-50K, teacher/student same-view knowledge-distillation
recipe based on the team's strongest proven architecture. Tail EMA weight
averaging is an additional *unverified* candidate, not a measured improvement.
"""

import copy
import csv
import dataclasses
import json
import hashlib
import math
import os
import random
import time
import zipfile
import io
from PIL import ImageFilter
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T


CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
LIMIT = 5_000_000


@dataclass
class TrainConfig:
    role: str = "student"
    data: str = ""                # provided data.zip or extracted dataset root
    teacher: str = ""             # Path to the teacher checkpoint
    init_student_path: str = ""   # optional earlier single student EMA checkpoint (fine-tune)
    output: str = ""              # empty = automatic writable output
    run_name: str = "A100_Full50K_KD_seed0"
    epochs: int = 300
    student_arch: str = "resnet18t"  # optional wrn34_3 challenger is also under 5M
    batch_size: int = 256
    workers: int = 8
    seed: int = 0
    val_per_class: int = 0       # 0 = all 50k; 500 = 45k/5k pilot
    lr: float = 0.20
    momentum: float = 0.90
    weight_decay: float = 5e-4
    warmup_epochs: int = 5
    ema_decay: float = 0.999
    kd_temperature: float = 4.0
    kd_alpha: float = 0.90       # matched-view baseline; proven at 88.6 in related run
    # Optional experiment: decrease KD weight during final epochs. Keep 0.90 for faithful baseline.
    kd_end_alpha: float = 0.90
    kd_decay_last_epochs: int = 80
    cutout: int = 16
    grayscale_p: float = 0.05
    # Tail averaging is an alternate single-student checkpoint, not an ensemble.
    tail_start_epoch: int = 255
    tail_every: int = 5
    save_every: int = 5
    tta_modes: str = "flip,multi,cardinal"
    auto_resume: bool = True
    quick_test: bool = False      # only for verifying the pipeline, no performance claims
    export_torchscript: bool = False
    # Robust-image augmentations are applied ONLY to permitted train images.
    aug_blur_p: float = 0.0
    aug_jpeg_p: float = 0.0
    aug_color_p: float = 0.0
    mixup_prob: float = 0.0
    mixup_alpha: float = 0.2
    mixup_end_epoch: int = 0
    sla: bool = False
    sla_joint_weight: float = 0.5
    sla_sd_weight: float = 0.5
    sla_warmup_epochs: int = 10
    sla_sd_temperature: float = 1.0


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_data_folder(root, max_depth=5):
    root = Path(root).expanduser()
    if not root.is_dir():
        return None
    for where, dirs, files in os.walk(root):
        d = Path(where)
        if 'train_labels.csv' in files and (d / 'train_images').is_dir() and (d / 'test_images').is_dir():
            return d
        if len(d.relative_to(root).parts) >= max_depth:
            dirs.clear()
    return None


def resolve_data(cfg, work):
    search_dirs = []
    if cfg.data:
        explicit = Path(cfg.data).expanduser()
        if not explicit.exists():
            raise FileNotFoundError(f"DATA not found: {explicit}")
        if explicit.is_dir():
            found = find_data_folder(explicit)
            if found: return found
            search_dirs.append(explicit)
        else:
            search_dirs.append(explicit)
    else:
        for p in [Path('/kaggle/input'), Path.cwd(), Path.home(), Path('/mnt/data')]:
            if p.exists(): search_dirs.append(p)
    for entry in search_dirs:
        if entry.is_dir():
            found = find_data_folder(entry, max_depth=5 if entry == Path('/kaggle/input') else 2)
            if found: return found
        elif entry.is_file() and zipfile.is_zipfile(entry):
            return unzip_data(entry, work / 'extracted_data')
    for root in search_dirs:
        if root.is_dir():
            candidates = list(root.glob('*.zip')) + list(root.glob('*/*.zip'))
            for entry in candidates:
                if zipfile.is_zipfile(entry):
                    with zipfile.ZipFile(entry) as z:
                        names = z.namelist()
                        if any(x.endswith('train_labels.csv') for x in names):
                            return unzip_data(entry, work / 'extracted_data')
    raise FileNotFoundError('Could not find provided dataset. Set data to data.zip or the folder containing train_labels.csv and train_images/.')


def unzip_data(zpath, output):
    found = find_data_folder(output)
    if found: return found
    output.mkdir(parents=True, exist_ok=True)
    print(f'Extracting supplied competition data: {zpath}', flush=True)
    with zipfile.ZipFile(zpath) as archive:
        archive.extractall(output)
    found = find_data_folder(output)
    if not found: raise FileNotFoundError('Dataset zip does not contain expected train_labels.csv / train_images / test_images.')
    return found


def resolve_teacher(path):
    p=Path(path).expanduser()
    if not path or not p.is_file(): raise FileNotFoundError(f'Teacher checkpoint missing: {path}. Run teacher stage first.')
    return p


def load_data_arrays(root, cache_dir):
    cache_dir.mkdir(parents=True, exist_ok=True)
    # Key the cache by the actual extracted dataset location to avoid silent cross-dataset reuse.
    key = hashlib.sha256(str(root.resolve()).encode()).hexdigest()[:10]
    cache_file = cache_dir / f'competition_arrays_{key}.npz'
    if cache_file.exists():
        with np.load(cache_file) as n:
            data = {k: n[k] for k in n.files}
        if len(data['labels']) == 50_000 and len(data['test_ids']) == 2_000:
            print(f'Loaded cached images: {cache_file}', flush=True)
            return data
    train_table = pd.read_csv(root / 'train_labels.csv').sort_values('id')
    test_table = pd.read_csv(root / 'test.csv').sort_values('id')
    assert {'id', 'filename', 'label'} <= set(train_table), 'train_labels.csv must contain id, filename, label'
    assert {'id', 'filename'} <= set(test_table), 'test.csv must contain id, filename'
    assert len(train_table) == 50_000 and len(test_table) == 2_000, (len(train_table), len(test_table))
    assert train_table['id'].is_unique and test_table['id'].is_unique
    train_files = [root / 'train_images' / f for f in train_table['filename']]
    test_files = [root / 'test_images' / f for f in test_table['filename']]
    def read_image(p):
        with Image.open(p) as im:
            return np.asarray(im.convert('RGB'), dtype=np.uint8)
    print('Decoding provided PNG images (one-time cache)...', flush=True)
    with ThreadPoolExecutor(max_workers=12) as pool:
        train = np.stack(list(pool.map(read_image, train_files)))
        test = np.stack(list(pool.map(read_image, test_files)))
    labels = train_table['label'].to_numpy(dtype=np.int64)
    assert train.shape == (50_000,32,32,3) and test.shape == (2_000,32,32,3)
    assert labels.min() == 0 and labels.max() == 9
    data = dict(train=train, labels=labels, test=test,
                test_ids=test_table['id'].to_numpy(dtype=np.int64))
    np.savez(cache_file, **data)
    print(f'Cached images: {cache_file}', flush=True)
    return data


def split_indices(labels, val_per_class, seed):
    if val_per_class <= 0: return np.arange(len(labels)), np.empty((0,),dtype=np.int64)
    rng = np.random.default_rng(seed)
    val = np.concatenate([rng.choice(np.flatnonzero(labels == c), size=val_per_class, replace=False) for c in range(10)])
    val.sort()
    train = np.setdiff1d(np.arange(len(labels)), val)
    return train, val


class Cutout:
    def __init__(self, size): self.size = size
    def __call__(self, x):
        _, h, w = x.shape
        y, xx = random.randrange(h), random.randrange(w)
        y0, y1 = max(0, y-self.size//2), min(h, y+self.size//2)
        x0, x1 = max(0, xx-self.size//2), min(w, xx+self.size//2)
        x[:,y0:y1,x0:x1] = 0.0
        return x


class CompetitionDataset(Dataset):
    """Training-only augmentations; test images never enter a training loader."""
    def __init__(self, images, labels, config):
        self.images, self.labels = images, labels
        self.config = config
        self.geom = T.Compose([
            T.RandomCrop(32, padding=4, padding_mode='reflect'),
            T.RandomHorizontalFlip(), T.TrivialAugmentWide(),
        ])
        self.gray=T.RandomGrayscale(p=config.grayscale_p)
        self.color=T.ColorJitter(brightness=.13, contrast=.15, saturation=.13)
        self.to_tensor=T.ToTensor()
        self.norm=T.Normalize(CIFAR_MEAN,CIFAR_STD)
        self.cut=Cutout(config.cutout)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        im=self.geom(Image.fromarray(self.images[i]))
        if random.random()<self.config.aug_blur_p:
            im=im.filter(ImageFilter.GaussianBlur(radius=random.uniform(.20,.65)))
        if random.random()<self.config.aug_jpeg_p:
            buf=io.BytesIO()
            im.save(buf, format='JPEG', quality=random.randint(65,90))
            buf.seek(0)
            with Image.open(buf) as tmp: im=tmp.convert('RGB')
        if random.random()<self.config.aug_color_p: im=self.color(im)
        im=self.gray(im)
        return self.cut(self.norm(self.to_tensor(im))), int(self.labels[i])


class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(cin,cout,3,stride=stride,padding=1,bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout,cout,3,padding=1,bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Identity()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin,cout,1,stride=stride,bias=False),nn.BatchNorm2d(cout))
    def forward(self,x):
        z = F.relu(self.bn1(self.conv1(x)))
        z = self.bn2(self.conv2(z))
        return F.relu(z + self.shortcut(x))


class TrimmedResNet18(nn.Module):
    def __init__(self, widths=(64,128,256,224), num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3,widths[0],3,padding=1,bias=False),nn.BatchNorm2d(widths[0]),nn.ReLU(inplace=True))
        stages=[]; cin=widths[0]
        for stage_idx, cout in enumerate(widths):
            blocks=[]
            for block_idx in range(2):
                stride = (2 if stage_idx > 0 else 1) if block_idx == 0 else 1
                blocks.append(BasicBlock(cin,cout,stride)); cin=cout
            stages.append(nn.Sequential(*blocks))
        self.stages = nn.Sequential(*stages)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(cin,num_classes)
        for module in self.modules():
            if isinstance(module,nn.Conv2d):
                nn.init.kaiming_normal_(module.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(module,nn.BatchNorm2d):
                nn.init.ones_(module.weight); nn.init.zeros_(module.bias)
        for module in self.modules():
            if isinstance(module,BasicBlock): nn.init.zeros_(module.bn2.weight)
    def forward(self,x):
        x = self.stages(self.stem(x))
        return self.classifier(self.pool(x).flatten(1))


class WideBasicBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.bn1=nn.BatchNorm2d(cin)
        self.conv1=nn.Conv2d(cin,cout,3,stride=stride,padding=1,bias=False)
        self.bn2=nn.BatchNorm2d(cout)
        self.conv2=nn.Conv2d(cout,cout,3,padding=1,bias=False)
        self.shortcut=None
        if stride!=1 or cin!=cout: self.shortcut=nn.Conv2d(cin,cout,1,stride=stride,bias=False)
    def forward(self,x):
        activated=F.relu(self.bn1(x))
        residual=self.conv2(F.relu(self.bn2(self.conv1(activated))))
        skip=x if self.shortcut is None else self.shortcut(activated)
        return residual+skip


class WideResNet(nn.Module):
    def __init__(self, depth=28, widen_factor=10, classes=10, sla=False):
        super().__init__()
        assert (depth-4)%6==0
        blocks=(depth-4)//6
        widths=[16,int(round(16*widen_factor)),int(round(32*widen_factor)),int(round(64*widen_factor))]
        self.stem=nn.Conv2d(3,16,3,padding=1,bias=False)
        groups=[]; cin=16
        for gi, cout in enumerate(widths[1:]):
            for bi in range(blocks):
                stride=(1 if gi==0 else 2) if bi==0 else 1
                groups.append(WideBasicBlock(cin,cout,stride));cin=cout
        self.groups=nn.Sequential(*groups)
        self.final_bn=nn.BatchNorm2d(cin)
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.classifier=nn.Linear(cin,classes)
        self.joint_fc=nn.Linear(cin,classes*4) if sla else None
    def forward(self,x,return_joint=False):
        x=F.relu(self.final_bn(self.groups(self.stem(x))))
        features=self.pool(x).flatten(1)
        logits=self.classifier(features)
        if return_joint:
            if self.joint_fc is None: raise ValueError('SLA head is disabled')
            return logits,self.joint_fc(features)
        return logits


def rotate_four_ways(images):
    # Interleaved: image 0 rotations 0,1,2,3; image 1 rotations 0,1,2,3; ...
    return torch.stack([torch.rot90(images,k,dims=(2,3)) for k in range(4)],dim=1).flatten(0,1).contiguous(memory_format=torch.channels_last)


def rotation_aggregate(joint_logits,batch_size):
    # (image, input rotation, class, output rotation)
    grouped=joint_logits.float().reshape(batch_size,4,10,4)
    return torch.stack([grouped[:,k,:,k] for k in range(4)]).mean(0)


def sla_auxiliary_loss(logits,joint_logits,labels,cfg,epoch):
    joint_labels=(4*labels[:,None]+torch.arange(4,device=labels.device)[None,:]).flatten()
    joint_ce=F.cross_entropy(joint_logits.float(),joint_labels)
    target=rotation_aggregate(joint_logits,labels.numel()).detach()
    t=cfg.sla_sd_temperature
    sd=F.kl_div(F.log_softmax(logits[::4].float()/t,dim=1),
                F.softmax(target/t,dim=1),reduction='batchmean')*(t*t)
    ramp=min(1.0,(epoch+1)/max(1,cfg.sla_warmup_epochs))
    return ramp*(cfg.sla_joint_weight*joint_ce+cfg.sla_sd_weight*sd)


class EMA:
    def __init__(self, model, decay):
        self.module=copy.deepcopy(model).eval()
        for p in self.module.parameters(): p.requires_grad_(False)
        self.decay=decay
        self.updates=0
    @torch.no_grad()
    def update(self,model):
        self.updates+=1
        decay=min(self.decay,(1+self.updates)/(10+self.updates))
        for target,src in zip(self.module.state_dict().values(),model.state_dict().values()):
            if target.dtype.is_floating_point: target.lerp_(src.detach(),1.0-decay)
            else: target.copy_(src)


class TailAverager:
    def __init__(self): self.sums={}; self.last_int={}; self.n=0
    @torch.no_grad()
    def add(self,model):
        state=model.state_dict()
        for key,tensor in state.items():
            x=tensor.detach().float().cpu()
            if tensor.dtype.is_floating_point:
                if key not in self.sums: self.sums[key]=x.clone()
                else: self.sums[key].add_(x)
            else: self.last_int[key]=tensor.detach().cpu().clone()
        self.n+=1
    def average_into(self,model):
        if self.n==0:return False
        current=model.state_dict()
        state={}
        for k,v in current.items():
            if v.dtype.is_floating_point: state[k]=(self.sums[k]/self.n).to(dtype=v.dtype)
            else: state[k]=self.last_int[k].to(dtype=v.dtype)
        model.load_state_dict(state,strict=True)
        return True
    def state_dict(self): return {'sums':self.sums,'last_int':self.last_int,'n':self.n}
    def load_state_dict(self,s):
        self.sums=s['sums'];self.last_int=s['last_int'];self.n=s['n']


def make_optimizer(model,cfg):
    decay=[];no_decay=[]
    for p in model.parameters():
        if p.requires_grad: (decay if p.ndim>1 else no_decay).append(p)
    return torch.optim.SGD([{'params':decay,'weight_decay':cfg.weight_decay},{'params':no_decay,'weight_decay':0.0}],
                           lr=cfg.lr,momentum=cfg.momentum,nesterov=True)


def schedule(optimizer,total_steps,warmup_steps):
    def factor(step):
        if step<warmup_steps: return (step+1)/max(1,warmup_steps)
        f=min(1.0,(step-warmup_steps)/max(1,total_steps-warmup_steps))
        return 0.5*(1.0+math.cos(math.pi*f))
    return torch.optim.lr_scheduler.LambdaLR(optimizer,factor)


def kd_alpha_for_epoch(cfg,epoch):
    if cfg.kd_decay_last_epochs<=0 or cfg.kd_alpha==cfg.kd_end_alpha:
        return cfg.kd_alpha
    start=max(0,cfg.epochs-cfg.kd_decay_last_epochs)
    t=min(1.0,max(0.0,(epoch-start)/max(1,cfg.epochs-start-1)))
    return cfg.kd_alpha+(cfg.kd_end_alpha-cfg.kd_alpha)*t


def normalize_all(images,device):
    x=torch.from_numpy(images.copy()).permute(0,3,1,2).float().div_(255.)
    mean=torch.tensor(CIFAR_MEAN).view(1,3,1,1)
    std=torch.tensor(CIFAR_STD).view(1,3,1,1)
    return ((x-mean)/std).contiguous(memory_format=torch.channels_last).to(device)


def teacher_from_checkpoint(path,device,cfg):
    ck=torch.load(path,map_location='cpu',weights_only=False)
    if ck.get('role')!='teacher' or ck.get('student_arch')!='wrn28_10':
        raise ValueError('Expected teacher_final_ema.pth generated by this notebook.')
    if not ck.get('training_complete'):
        raise ValueError('Teacher training is incomplete.')
    if ck.get('quick_test') and not cfg.quick_test:
        raise ValueError('Smoke-test teacher cannot be used for a full student run.')
    if ck.get('config',{}).get('val_per_class')!=cfg.val_per_class:
        raise ValueError('Teacher/student validation split settings differ.')
    teacher=WideResNet(28,10)
    teacher.load_state_dict(ck['ema_state'],strict=True)
    teacher=teacher.to(device).to(memory_format=torch.channels_last).eval()
    for p in teacher.parameters(): p.requires_grad_(False)
    return teacher


def view_logits(model,images,mode,amp_dtype):
    def forward(x):
        with torch.autocast(device_type='cuda',dtype=amp_dtype):
            return model(x).float()
    if mode=='none':return forward(images)
    def flip(x):return 0.5*(forward(x)+forward(x.flip(-1)))
    if mode=='flip':return flip(images)
    padded=F.pad(images,(1,1,1,1),mode='reflect')
    if mode=='multi':
        # 6 views: center+flip, diagonal -1/-1 and +1/+1 with flips
        a=padded[:,:,0:32,0:32]
        b=padded[:,:,2:34,2:34]
        return 0.5*flip(images)+0.25*flip(a)+0.25*flip(b)
    if mode=='cardinal':
        # 10 views: center+4 cardinal translations, each horizontal mirrored
        views=[images,padded[:,:,0:32,1:33],padded[:,:,2:34,1:33],
               padded[:,:,1:33,0:32],padded[:,:,1:33,2:34]]
        return sum(flip(v) for v in views)/5
    raise ValueError(f'Unknown TTA mode {mode}')


@torch.inference_mode()
def predict(model,inputs,mode,dtype,batch=256):
    model.eval()
    outs=[]
    for x in inputs.split(batch):
        outs.append(view_logits(model,x,mode,dtype).cpu())
    return torch.cat(outs)


def atomic_save(obj,path):
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(obj,tmp)
    os.replace(tmp,path)


def save_checkpoint(path,cfg,epoch,model,ema,optimizer,scheduler,scaler,averager,history):
    atomic_save({
        'config':dataclasses.asdict(cfg),'epoch':epoch,'model_state':model.state_dict(),
        'ema_state':ema.module.state_dict(),'ema_updates':ema.updates,
        'opt_state':optimizer.state_dict(),'sched_state':scheduler.state_dict(),
        'scaler_state':scaler.state_dict(),'tail_state':averager.state_dict(),
        'history':history},path)


def read_checkpoint(path,cfg,model,ema,optimizer,scheduler,scaler,averager):
    ck=torch.load(path,map_location='cpu',weights_only=False)
    old=ck['config']
    critical=['role','student_arch','epochs','batch_size','seed','val_per_class','lr','warmup_epochs','kd_alpha','kd_end_alpha',
              'kd_decay_last_epochs','tail_start_epoch','tail_every','cutout','grayscale_p','init_student_path',
               'aug_blur_p','aug_jpeg_p','aug_color_p','mixup_prob','mixup_alpha','mixup_end_epoch',
               'sla','sla_joint_weight','sla_sd_weight','sla_warmup_epochs','sla_sd_temperature',
               'kd_temperature','weight_decay','momentum','ema_decay','teacher']
    differences=[k for k in critical if old.get(k)!=getattr(cfg,k)]
    if differences:raise ValueError(f'Resume checkpoint has conflicting settings: {differences}. Change output or run_name.')
    model.load_state_dict(ck['model_state'])
    ema.module.load_state_dict(ck['ema_state']);ema.updates=ck['ema_updates']
    optimizer.load_state_dict(ck['opt_state']);scheduler.load_state_dict(ck['sched_state'])
    if ck.get('scaler_state'):scaler.load_state_dict(ck['scaler_state'])
    averager.load_state_dict(ck['tail_state'])
    return ck['epoch']+1,ck['history']


def run(cfg:TrainConfig):
    start=time.time()
    if cfg.sla:
        assert cfg.student_arch.startswith('wrn'), 'SLA is implemented for WRN students'
        assert cfg.mixup_prob==0, 'Disable Mixup for this first controlled SLA experiment'
        assert cfg.sla_sd_temperature>0
        assert cfg.sla_joint_weight>=0 and cfg.sla_sd_weight>=0
    assert cfg.role in ('teacher','student')
    if cfg.role=='teacher':
        assert cfg.student_arch=='wrn28_10' and not cfg.sla and not cfg.teacher
        assert not cfg.init_student_path
    seed_all(cfg.seed)
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU required. Request 1x A100 or Kaggle GPU before running.')
    device=torch.device('cuda:0')
    gpu=torch.cuda.get_device_name(0)
    dtype=torch.bfloat16 if torch.cuda.get_device_capability(0)[0]>=8 else torch.float16
    torch.backends.cudnn.benchmark=True
    torch.backends.cuda.matmul.allow_tf32=True
    torch.backends.cudnn.allow_tf32=True
    print(f'GPU: {gpu} | precision: {dtype} | PyTorch {torch.__version__}',flush=True)
    if cfg.output:
        out_base=Path(cfg.output).expanduser()
    else:
        out_base=Path('/kaggle/working/cifar_champion') if Path('/kaggle/working').is_dir() else Path.home()/'cifar_champion'
    run_dir=out_base/(cfg.run_name+('_quick' if cfg.quick_test else ''))
    cache_dir=Path('/content/operation90_cache')
    run_dir.mkdir(parents=True,exist_ok=True)
    print(f'Output: {run_dir}',flush=True)
    root=resolve_data(cfg,cache_dir)
    teacher_path=resolve_teacher(cfg.teacher) if cfg.role=='student' else None
    print(f'Dataset: {root} | role={cfg.role} | teacher={teacher_path}',flush=True)
    arr=load_data_arrays(root,cache_dir)
    train_ids,val_ids=split_indices(arr['labels'],cfg.val_per_class,42)
    requested_epochs=cfg.epochs
    if cfg.quick_test:
        train_ids=train_ids[:1024];val_ids=val_ids[:min(500,len(val_ids))]
        cfg.epochs=2; cfg.warmup_epochs=1;cfg.batch_size=128
    print(f'Train images={len(train_ids)}; honest holdout={len(val_ids)}; epochs={cfg.epochs}',flush=True)
    train_ds=CompetitionDataset(arr['train'][train_ids],arr['labels'][train_ids],cfg)
    worker_count=min(cfg.workers,max(0,(os.cpu_count() or 2)-1))
    loader_args=dict(batch_size=cfg.batch_size,shuffle=True,drop_last=True,
                     num_workers=worker_count,pin_memory=True)
    if worker_count>0:loader_args.update(persistent_workers=True,prefetch_factor=2)
    train_loader=DataLoader(train_ds,**loader_args)
    val_x=normalize_all(arr['train'][val_ids],device) if len(val_ids) else None
    val_y=arr['labels'][val_ids] if len(val_ids) else None
    test_x=normalize_all(arr['test'],device)
    if cfg.student_arch == 'wrn28_10' and cfg.role=='teacher':
        model = WideResNet(28,10)
    elif cfg.student_arch == 'resnet18t':
        model = TrimmedResNet18()
    elif cfg.student_arch == 'wrn34_3':
        model = WideResNet(34,3,sla=cfg.sla)
    elif cfg.student_arch == 'wrn34_325':
        model = WideResNet(34,3.25,sla=cfg.sla)
    else:
        raise ValueError('Student architecture must be resnet18t, wrn34_3, or wrn34_325')
    model = model.to(device).to(memory_format=torch.channels_last)
    if cfg.init_student_path:
        init_file = Path(cfg.init_student_path).expanduser()
        if not init_file.is_file():
            raise FileNotFoundError(f'Initial student checkpoint missing: {init_file}')
        initial = torch.load(init_file, map_location='cpu', weights_only=False)
        if initial.get('student_arch') != cfg.student_arch or 'ema_state' not in initial:
            raise ValueError('Initial checkpoint must be a matching single-student EMA export.')
        # Permit only the newly added joint head to be absent in a legacy champion.
        incompatible=model.load_state_dict(initial['ema_state'],strict=False)
        allowed={'joint_fc.weight','joint_fc.bias'} if cfg.sla else set()
        if set(incompatible.missing_keys)-allowed or incompatible.unexpected_keys:
            raise ValueError(f'Incompatible initial checkpoint: {incompatible}')
        print(f'FINE-TUNE: loaded previous EMA student from {init_file}', flush=True)
        del initial
    nparams=sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'{cfg.role.upper()}: {cfg.student_arch}, {nparams:,} parameters; student limit {LIMIT:,}',flush=True)
    if cfg.role=='student': assert nparams<=LIMIT
    ema=EMA(model,cfg.ema_decay)
    teacher=teacher_from_checkpoint(teacher_path,device,cfg) if teacher_path else None
    print('Frozen WRN-28-10 teacher loaded.' if teacher is not None else 'Training teacher from random initialization with correct labels only.',flush=True)
    optimizer=make_optimizer(model,cfg)
    steps=len(train_loader)
    scheduler=schedule(optimizer,cfg.epochs*steps,cfg.warmup_epochs*steps)
    scaler=torch.amp.GradScaler('cuda',enabled=(dtype==torch.float16))
    averager=TailAverager();history=[];start_epoch=0
    last_ckpt=run_dir/'last.pth'
    if cfg.auto_resume and last_ckpt.exists():
        start_epoch,history=read_checkpoint(last_ckpt,cfg,model,ema,optimizer,scheduler,scaler,averager)
        print(f'Resumed at epoch {start_epoch+1}/{cfg.epochs}',flush=True)
    if teacher is not None: print('Paired same-view teacher distillation plus optional rotation self-distillation.',flush=True)
    print(f'KD T={cfg.kd_temperature}; alpha {cfg.kd_alpha}->{cfg.kd_end_alpha}; EMA={cfg.ema_decay}',flush=True)
    print(f'SLA={cfg.sla}; joint={cfg.sla_joint_weight}; self-KD={cfg.sla_sd_weight}. Inference uses normal head + existing TTA.',flush=True)
    if cfg.sla: print('Four student views per training image; teacher uses only the unrotated augmented view. Measure first-epoch runtime.',flush=True)
    for epoch in range(start_epoch,cfg.epochs):
        t0=time.time();model.train()
        loss_sum=0.;correct=0;seen=0
        alpha=kd_alpha_for_epoch(cfg,epoch)
        for x,y in train_loader:
            x=x.to(device,non_blocking=True).to(memory_format=torch.channels_last)
            y=y.to(device,non_blocking=True)
            # Optional low-probability MixUp in the early/mid epochs.
            do_mix=(cfg.mixup_prob>0 and epoch<cfg.mixup_end_epoch and
                    random.random()<cfg.mixup_prob)
            if do_mix:
                perm=torch.randperm(y.numel(),device=device)
                lam=float(np.random.beta(cfg.mixup_alpha,cfg.mixup_alpha))
                x=x*lam+x[perm]*(1-lam)
            t_logits=None
            if teacher is not None:
                with torch.no_grad(),torch.autocast(device_type='cuda',dtype=dtype):
                    t_logits=teacher(x).float()
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=dtype):
                if cfg.sla:
                    all_logits,joint_logits=model(rotate_four_ways(x),return_joint=True)
                    s_logits=all_logits[::4]
                else:
                    s_logits=model(x)
            sl=s_logits.float()
            t=cfg.kd_temperature
            kd=sl.new_zeros(())
            if t_logits is not None:
                kd=F.kl_div(F.log_softmax(sl/t,dim=1),F.softmax(t_logits/t,dim=1),reduction='batchmean')*(t*t)
            if do_mix:
                ce=lam*F.cross_entropy(sl,y)+(1-lam)*F.cross_entropy(sl,y[perm])
            else:
                ce=F.cross_entropy(sl,y)
            loss=alpha*kd+(1-alpha)*ce if teacher is not None else ce
            if cfg.sla:
                loss=loss+sla_auxiliary_loss(all_logits,joint_logits,y,cfg,epoch)
            scaler.scale(loss).backward()
            scaler.step(optimizer);scaler.update();scheduler.step()
            ema.update(model)
            bs=y.numel();seen+=bs
            loss_sum+=loss.detach().item()*bs
            pred=s_logits.argmax(1)
            if do_mix:
                correct+=lam*(pred==y).sum().item()+(1-lam)*(pred==y[perm]).sum().item()
            else:
                correct+=(pred==y).sum().item()
        v_acc=None;v_ema=None
        if val_x is not None:
            yhat=predict(model,val_x,'none',dtype,batch=256)
            ehat=predict(ema.module,val_x,'none',dtype,batch=256)
            v_acc=float((yhat.argmax(1).numpy()==val_y).mean()*100)
            v_ema=float((ehat.argmax(1).numpy()==val_y).mean()*100)
        ep=epoch+1
        if ep>=cfg.tail_start_epoch and ep%cfg.tail_every==0:
            averager.add(ema.module)
        elapsed=time.time()-t0
        row={'epoch':ep,'lr':optimizer.param_groups[0]['lr'],'kd_alpha':alpha,
             'loss':loss_sum/max(1,seen),'train_aug_acc':100*correct/max(1,seen),
             'val_acc':v_acc,'val_ema':v_ema,'epoch_seconds':elapsed,'tail_snapshots':averager.n}
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir/'history.csv',index=False)
        val_info=f' | val {v_acc:.2f}% | EMA {v_ema:.2f}%' if v_acc is not None else ' | full50k (no honest validation)'
        print(f"epoch {ep:3d}/{cfg.epochs} | lr {row['lr']:.4f} | alpha {alpha:.2f} | loss {row['loss']:.3f} | train {row['train_aug_acc']:.2f}%"+
              val_info+f' | {elapsed:.1f}s',flush=True)
        if ep==start_epoch+1:
            print(f'Estimated remaining at current speed: {elapsed*(cfg.epochs-ep)/3600:.2f}h',flush=True)
            if cfg.quick_test: print(f'Smoke test uses a tiny subset. Its runtime does NOT estimate the full {requested_epochs}-epoch run.',flush=True)
        if ep%cfg.save_every==0 or ep==cfg.epochs:
            save_checkpoint(last_ckpt,cfg,epoch,model,ema,optimizer,scheduler,scaler,averager,history)
            print(f'  Checkpoint saved: {last_ckpt} (epoch {ep})',flush=True)
    if cfg.role=='teacher':
        teacher_export=run_dir/'teacher_final_ema.pth'
        atomic_save({'role':'teacher','student_arch':'wrn28_10','nparams':nparams,
                     'ema_state':ema.module.state_dict(),'config':dataclasses.asdict(cfg),
                     'training_complete':len(history)==cfg.epochs,'quick_test':cfg.quick_test},teacher_export)
        print(f'TEACHER SAVED: {teacher_export}. No teacher prediction CSV is exported.',flush=True)
        return {'role':'teacher','checkpoint':str(teacher_export),'params':nparams}
    # Final and tail-averaged student are each single <5M-parameter networks.
    atomic_save({'student_arch':cfg.student_arch,'nparams':nparams,'ema_state':ema.module.state_dict(),
                 'config':dataclasses.asdict(cfg)},run_dir/'student_final_ema.pth')
    candidates={'final_ema':ema.module}
    if averager.n:
        avg=copy.deepcopy(ema.module)
        averager.average_into(avg)
        candidates['tail_avg']=avg
        atomic_save({'student_arch':cfg.student_arch,'nparams':nparams,'ema_state':avg.state_dict(),
                     'n_snapshots':averager.n,'config':dataclasses.asdict(cfg)},run_dir/'student_tail_avg.pth')
    if cfg.init_student_path:
        # Linear mode connectivity: same student trained further, NOT a two-model ensemble.
        init_obj = torch.load(Path(cfg.init_student_path).expanduser(), map_location='cpu', weights_only=False)
        init_state = init_obj['ema_state']
        blend = copy.deepcopy(ema.module)
        state = blend.state_dict()
        for k, tensor in state.items():
            if tensor.dtype.is_floating_point:
                # A legacy champion has no joint head: retain the learned new head.
                if k in init_state:
                    earlier = init_state[k].to(device=tensor.device, dtype=tensor.dtype)
                    tensor.mul_(0.70).add_(earlier, alpha=0.30)
        blend.load_state_dict(state, strict=True)
        candidates['origblend30'] = blend
        atomic_save({'student_arch':cfg.student_arch,'nparams':nparams,
                     'ema_state':blend.state_dict(),'config':dataclasses.asdict(cfg)},
                    run_dir/'student_origblend30.pth')
        del init_obj, init_state
    if cfg.quick_test:
        print('Smoke test complete; checkpoint saved, no submission predictions exported.',flush=True)
        return {'quick_test':True,'params':nparams}
    modes=[s.strip() for s in cfg.tta_modes.split(',') if s.strip()]
    report={'params':nparams,'teacher_path':str(teacher_path),'train_size':len(train_ids),
            'holdout_size':len(val_ids),'tail_snapshots':averager.n,'runtime_hours':(time.time()-start)/3600,
            'candidates':{}}
    sample_path=root/'sample_submission.csv'
    if sample_path.is_file():
        sample=pd.read_csv(sample_path)
        assert set(sample.columns)=={'id','label'}
        assert set(sample['id'])==set(arr['test_ids']), 'Competition sample IDs do not match data IDs'
    for candidate_name,candidate in candidates.items():
        report['candidates'][candidate_name]={}
        for mode in modes:
            logits=predict(candidate,test_x,mode,dtype,batch=256)
            preds=logits.argmax(1).numpy()
            if len(preds)!=len(arr['test_ids']):raise RuntimeError('Incorrect test-prediction count')
            output=pd.DataFrame({'id':arr['test_ids'],'label':preds})
            output.to_csv(run_dir/f'submission_{candidate_name}_{mode}.csv',index=False)
            np.save(run_dir/f'test_logits_{candidate_name}_{mode}.npy',logits.numpy().astype(np.float32))
            result={'csv':f'submission_{candidate_name}_{mode}.csv'}
            if val_x is not None:
                scores=predict(candidate,val_x,mode,dtype,batch=256)
                result['val_accuracy']=float((scores.argmax(1).numpy()==val_y).mean()*100)
            report['candidates'][candidate_name][mode]=result
            print(f"READY: {result['csv']}"+(f" | heldout {result['val_accuracy']:.2f}%" if 'val_accuracy' in result else ''),flush=True)
    if cfg.export_torchscript:
        # The teacher is NOT exported. This exports the <5M-parameter student only.
        final_copy=copy.deepcopy(ema.module).float().cpu().eval().to(memory_format=torch.contiguous_format)
        scripted=torch.jit.trace(final_copy,torch.randn(1,3,32,32))
        scripted.save(str(run_dir/'student_final_ema_scripted.pt'))
    with open(run_dir/'report.json','w') as f:json.dump(report,f,indent=2)
    print(f'FINISHED. Files in {run_dir}',flush=True)
    print('Primary single-model candidate: submission_final_ema_multi.csv',flush=True)
    print('Alternative same-size student: submission_tail_avg_multi.csv',flush=True)
    return report


# Notebook mode: configure in the next cell and call run(CFG) in the final cell.

print("Setup complete. Connecting Drive and locating your data...", flush=True)


import zipfile
from pathlib import Path
from dataclasses import replace
import shutil

QUICK_TEST = False
STAGE = 'both'  # 'teacher', 'student', or 'both'
VAL_PER_CLASS = 500  # 500: honest pilot; 0: final full50k recipe (new folders, retrain both)
DATA_OVERRIDE = ''
TEACHER_CHECKPOINT = ''  # Only for STAGE='student' in a later session.
RESUME_SOURCE_DIR = ''  # Optional saved stage folder; STAGE must be teacher or student.
from google.colab import drive, files
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU, then rerun.'
drive.mount('/content/drive')
OUTROOT = Path('/content/drive/MyDrive/Operation90_TwoStage')
# Keep PNG decoding/extraction on local disk, not Drive (52,000 small files).
INPUT_ROOT = Path('/content/operation90_input')
INPUT_ROOT.mkdir(parents=True,exist_ok=True)
if not DATA_OVERRIDE:
    local_zip=INPUT_ROOT/'data-DL.zip'
    drive_zip=Path('/content/drive/MyDrive/data-DL.zip')
    if not local_zip.is_file():
        if drive_zip.is_file():
            shutil.copy2(drive_zip,local_zip)
        else:
            print('Select data-DL.zip in the upload prompt.',flush=True)
            old_cwd=Path.cwd()
            try:
                os.chdir(INPUT_ROOT)
                uploaded=files.upload()
                del uploaded
            finally:
                os.chdir(old_cwd)
            if not local_zip.is_file():
                raise FileNotFoundError('Upload the file named data-DL.zip, or set DATA_OVERRIDE to its actual path.')
    DATA_OVERRIDE=str(local_zip)
print('Persistent checkpoints:',OUTROOT,flush=True)
assert STAGE in ('teacher','student','both')
assert VAL_PER_CLASS in (0,500)
def locate_input_data(input_root, override=''):
    if override:
        path=Path(override).expanduser()
        if not path.exists():
            raise FileNotFoundError(f'DATA_OVERRIDE does not exist: {path}. Leave it empty for automatic detection.')
        return path
    # Prefer your named ZIP. Never guess among multiple copies.
    archives=sorted(input_root.rglob('data-DL.zip'))
    if len(archives)>1:
        raise RuntimeError(f'Multiple data-DL.zip files found: {archives}. Set DATA_OVERRIDE to the intended one.')
    if len(archives)==1:
        if not zipfile.is_zipfile(archives[0]): raise ValueError('data-DL.zip is not a valid ZIP archive.')
        return archives[0]
    folders=sorted({p.parent for p in input_root.rglob('train_labels.csv')
                    if (p.parent/'test.csv').is_file()
                    and (p.parent/'train_images').is_dir()
                    and (p.parent/'test_images').is_dir()})
    if len(folders)!=1:
        raise RuntimeError(f'Attach data-DL.zip using Add Input, or attach one extracted competition dataset. Found folders: {folders}')
    return folders[0]

DATA=locate_input_data(INPUT_ROOT,DATA_OVERRIDE)
print('DATA FOUND:',DATA,flush=True)
SPLIT_TAG='val45k' if VAL_PER_CLASS else 'full50k'
TEACHER_CFG=TrainConfig(
    role='teacher',data=str(DATA),teacher='',student_arch='wrn28_10',
    run_name=f'Teacher_WRN28_10_{SPLIT_TAG}_seed17', output=str(OUTROOT),
    epochs=300,batch_size=128,workers=4,seed=17,val_per_class=VAL_PER_CLASS,
    lr=.1,momentum=.9,weight_decay=5e-4,warmup_epochs=5,ema_decay=.999,
    cutout=14,grayscale_p=.05,mixup_prob=0.,sla=False,
    tail_start_epoch=255,tail_every=5,save_every=1,
    quick_test=QUICK_TEST,export_torchscript=False)
teacher_folder=OUTROOT/(TEACHER_CFG.run_name+('_quick' if QUICK_TEST else ''))
DEFAULT_TEACHER=teacher_folder/'teacher_final_ema.pth'
STUDENT_CFG=TrainConfig(
    role='student',data=str(DATA),teacher=str(DEFAULT_TEACHER),
    student_arch='wrn34_325',run_name=f'Student_WRN34_325_KD_SLA_{SPLIT_TAG}_seed41',
    output=str(OUTROOT),epochs=330,batch_size=64,workers=4,seed=41,
    val_per_class=VAL_PER_CLASS,lr=.04625,momentum=.9,weight_decay=5e-4,
    warmup_epochs=5,ema_decay=.999,kd_temperature=4.,kd_alpha=.88,kd_end_alpha=.60,
    kd_decay_last_epochs=120,cutout=14,grayscale_p=.05,
    aug_blur_p=.045,aug_jpeg_p=.025,aug_color_p=.06,mixup_prob=0.,
    sla=True,sla_joint_weight=.5,sla_sd_weight=.5,sla_warmup_epochs=10,
    tail_start_epoch=280,tail_every=5,save_every=1,
    tta_modes='multi,flip,cardinal',quick_test=QUICK_TEST)
if TEACHER_CHECKPOINT:
    assert STAGE=='student', 'Use TEACHER_CHECKPOINT only with STAGE=student.'
    STUDENT_CFG=replace(STUDENT_CFG,teacher=str(Path(TEACHER_CHECKPOINT)))
if RESUME_SOURCE_DIR:
    assert STAGE!='both', 'Restore one stage at a time.'
    stage_cfg=TEACHER_CFG if STAGE=='teacher' else STUDENT_CFG
    destination=OUTROOT/(stage_cfg.run_name+('_quick' if QUICK_TEST else ''))
    if destination.exists(): raise RuntimeError(f'Resume destination exists: {destination}. Do not overwrite it.')
    source=Path(RESUME_SOURCE_DIR)
    assert (source/'last.pth').is_file(), 'Saved folder must contain last.pth.'
    shutil.copytree(source,destination)
print('Data:',DATA)
print('Stage:',STAGE,'| smoke test:',QUICK_TEST,'| validation per class:',VAL_PER_CLASS)
print('Outputs:',OUTROOT)


from zipfile import ZipFile,ZIP_DEFLATED
import gc

def package_student(cfg):
    folder=Path(cfg.output)/(cfg.run_name+('_quick' if cfg.quick_test else ''))
    csvs=sorted(folder.glob('submission_*.csv'))
    if not csvs: raise RuntimeError('No student prediction CSVs were generated.')
    archive=folder/'student_submissions.zip'
    with ZipFile(archive,'w',ZIP_DEFLATED) as z:
        for file in csvs+[folder/'report.json',folder/'history.csv']:
            if file.is_file(): z.write(file,file.name)
    print('Prediction ZIP:',archive)
    print('Preserve weights separately:',folder/'student_final_ema.pth')

if STAGE in ('teacher','both'):
    TEACHER_REPORT=run(TEACHER_CFG)
    print('Save this teacher checkpoint:',TEACHER_REPORT['checkpoint'])
    gc.collect()
    torch.cuda.empty_cache()
if STAGE in ('student','both'):
    if not Path(STUDENT_CFG.teacher).is_file():
        raise FileNotFoundError('Complete teacher training first, or attach its teacher_final_ema.pth and set TEACHER_CHECKPOINT.')
    STUDENT_REPORT=run(STUDENT_CFG)
    if not QUICK_TEST: package_student(STUDENT_CFG)
if QUICK_TEST:
    print('Two-epoch smoke test finished. No submission CSVs. Set QUICK_TEST=False for real training.')


Setup complete. Connecting Drive and locating your data...
Mounted at /content/drive
Select data-DL.zip in the upload prompt.


Saving data-DL.zip to data-DL.zip
Persistent checkpoints: /content/drive/MyDrive/Operation90_TwoStage
DATA FOUND: /content/operation90_input/data-DL.zip
Data: /content/operation90_input/data-DL.zip
Stage: both | smoke test: False | validation per class: 500
Outputs: /content/drive/MyDrive/Operation90_TwoStage
GPU: NVIDIA A100-SXM4-40GB | precision: torch.bfloat16 | PyTorch 2.11.0+cu130
Output: /content/drive/MyDrive/Operation90_TwoStage/Teacher_WRN28_10_val45k_seed17
Extracting supplied competition data: /content/operation90_input/data-DL.zip
Dataset: /content/operation90_cache/extracted_data | role=teacher | teacher=None
Decoding provided PNG images (one-time cache)...
Cached images: /content/operation90_cache/competition_arrays_ba2fd85c06.npz
Train images=45000; honest holdout=5000; epochs=300
TEACHER: wrn28_10, 36,479,194 parameters; student limit 5,000,000
Training teacher from random initialization with correct labels only.
KD T=4.0; alpha 0.9->0.9; EMA=0.999
SLA=False; joint=0.5;